# RAG from Scratch to Production — Bonus Episode
## Fixing it in production

Episode 18 attacked a copy of the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s auth code and found two real gaps:
1. **Conversations had no owner.** Anyone holding another user's `conversation_id` got that user's history pulled into their own request.
2. **Chat had no rate limit.** One user could trigger unlimited LLM calls, and spend a shared free-tier quota for everyone.

The notebooks showed the fixes on a mini app. This bonus episode applies them to the **real app**, tests them with its own test suite, and attacks the real app again, before and after.

**This one doesn't run in Colab:** it sends requests to a running copy of the app. To reproduce it, start the app with `docker compose up` (see the app's README) and point the notebook at it.

## 1. The fix

### Conversations get an owner

`app/services/conversation.py`, a new `resolve_conversation()` that every chat now goes through **before** reading or writing history:

```python
def resolve_conversation(self, conversation_id, user_id):
    if not conversation_id or conversation_id == "string":
        return self.create_conversation(user_id)          # new conversation, owned by this user

    owner = self.store.get_owner(conversation_id)
    if owner is None:
        # Unknown id: the store is in memory, so after a restart every id is unknown,
        # and the frontend keeps sending the one it has. The history is already gone.
        self.store.create(conversation_id)
        self.store.set_owner(conversation_id, user_id)
    elif owner != user_id:
        raise ConversationAccessDenied(conversation_id)
    return conversation_id
```

`app/api/chat.py` turns that error into **404 "Conversation not found"**, not 403, so the response doesn't confirm the conversation exists (Episode 18's 403-vs-404 lesson).

### Chat gets a per-user rate limit

```python
_chat_rate_limiter = InMemoryRateLimiter(
    max_attempts=settings.chat_rate_limit_requests,          # default 20
    window_seconds=settings.chat_rate_limit_window_seconds,  # default 60
)

def _limit_chat_per_user(current_user: User = Depends(get_current_user)):
    _chat_rate_limiter.check(f"user:{current_user.id}")

@router.post("", dependencies=[Depends(_limit_chat_per_user)])         # /chat
@router.post("/stream", dependencies=[Depends(_limit_chat_per_user)])  # /chat/stream
```

Keyed by **user id**, not IP (so coworkers on one network don't share a budget), and shared by `/chat` and `/chat/stream` (so switching endpoints doesn't double the allowance). The limits are settings, so they can be tuned per deployment.

### Tests

The app's own suite (the same command its CI runs) now includes:
- `tests/test_conversation_ownership.py`: a creator keeps their conversation, another user is refused (and nothing is added to or read from it), an unknown id is claimed, and Swagger's `"string"` placeholder starts a new one.
- `tests/test_chat_api.py`: another user's conversation returns 404, and chat is rate-limited per user across both endpoints, while a second user is unaffected.

Result: **29 passed**: the 23 existing tests plus 6 new ones.

## 2. Attack the real app, before and after

Set `BEFORE_URL` to a running copy of the app **without** the fix, and `AFTER_URL` to one **with** it. Either can be left empty. For a quick demo, the "after" server below runs with `CHAT_RATE_LIMIT_REQUESTS=3` (the default is 20).

In [1]:
import os, uuid, requests

BEFORE_URL = os.getenv("BEFORE_URL")   # e.g. http://localhost:8000 running master
AFTER_URL = os.getenv("AFTER_URL")     # e.g. http://localhost:8001 running the fix


def register(base, tag):
    email, password = f"{tag}-{uuid.uuid4().hex[:8]}@example.com", "TestPassword123"
    requests.post(f"{base}/auth/register", json={"email": email, "password": password}).raise_for_status()
    token = requests.post(f"{base}/auth/login", json={"email": email, "password": password}).json()["access_token"]
    return {"Authorization": f"Bearer {token}"}


def attack(base, label):
    if not base:
        print(f"══ {label}: skipped (no URL set)\n")
        return
    alice, bob = register(base, "alice"), register(base, "bob")
    first = requests.post(f"{base}/chat", headers=alice, timeout=300,
                          json={"question": "I'm on a performance improvement plan. When is my salary review?"})
    conversation_id = first.json()["conversation_id"]

    reuse = requests.post(f"{base}/chat", headers=bob, timeout=300,
                          json={"question": "Does that apply to me too?", "conversation_id": conversation_id})
    burst = [requests.post(f"{base}/chat", headers=bob, timeout=300, json={"question": f"question {i}"}).status_code
             for i in range(5)]

    print(f"══ {label}")
    print(f"   Alice starts a private conversation        → {first.status_code}")
    print(f"   Bob sends Alice's conversation_id          → {reuse.status_code} {reuse.json().get('detail', '')}")
    print(f"   Bob sends 5 chats in a row                 → {burst}\n")


attack(BEFORE_URL, "BEFORE: production without the fix")
attack(AFTER_URL, "AFTER: production with the fix")

══ BEFORE: production without the fix
   Alice starts a private conversation        → 200
   Bob sends Alice's conversation_id          → 200 
   Bob sends 5 chats in a row                 → [200, 200, 200, 200, 200]


══ AFTER: production with the fix
   Alice starts a private conversation        → 200
   Bob sends Alice's conversation_id          → 404 Conversation not found
   Bob sends 5 chats in a row                 → [200, 200, 429, 429, 429]


**Before:** the real app accepts Bob's request inside Alice's conversation, and serves every chat.

The visible answer doesn't show the leak, because the prompt rules stop the model from repeating history as fact. The server log does. Bob asked only *"Does that apply to me too?"*, and the query rewriter (Episode 9) used **Alice's** conversation to turn it into:

```
Original:  Does that apply to me too?
Rewritten: "Is a performance improvement plan salary review applicable to me?"
```

Alice's private situation became Bob's search query. *(From the server log of the "before" run while recording this episode.)*

**After:** Bob gets **404 Conversation not found**, before anything is read from or written to Alice's conversation, and his chats are cut off with **429** once he reaches the limit.

## 3. Design decisions worth copying

| Decision | Why |
|---|---|
| Check ownership **before** touching history | A check after the history is read would already have leaked it |
| 404, not 403 | Don't confirm that someone else's conversation exists |
| Claim unknown ids | The store is in memory: after a restart, real users would otherwise get errors for their own old conversations |
| Rate limit keyed by user, not IP | Follows the account, and doesn't lock out coworkers on one network (Episode 18) |
| One budget for `/chat` and `/chat/stream` | Switching endpoints shouldn't double the allowance |
| Limits as settings | Each deployment (free tier vs paid) can set its own |

## 4. Still open

This fix closes the two gaps that Episode 18 published. From the same episode:
- The **login** limiter still counts successful logins and is keyed by IP only.
- `/upload` isn't rate-limited yet.
- Document endpoints still return 403 (not 404) for other users' documents.
- Conversations and rate-limit counters live in memory: they reset on restart and aren't shared between processes.

And from the rest of the series: the BM25 tokenizer (Ep 7), hybrid scaling (Ep 8), the parser (Ep 3), the blocking event loop (Ep 17) and the skipped CI tests (Ep 20).

**The rule:** a vulnerability you've published is a deadline. Fix it, test it on the real system, and say so.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)